In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp05
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/mlfp05/ex_1.py ──
"""
Shared infrastructure for Exercise 1 — The Complete Autoencoder Family.

Contains: data loading, visualisation helpers, training loop, engine setup.
Technique-specific code does NOT belong here.
"""

import asyncio
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision

from kailash.db import ConnectionManager
from kailash_ml import ExperimentTracker
from kailash_ml import ModelRegistry


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()
torch.manual_seed(42)
np.random.seed(42)
device = get_device()

# Output directory for all visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex1_autoencoders"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Fashion-MNIST (full 60K)
# ════════════════════════════════════════════════════════════════════════

REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data" / "mlfp05" / "fashion_mnist"
DATA_DIR.mkdir(parents=True, exist_ok=True)

INPUT_DIM = 28 * 28
LATENT_DIM = 16
EPOCHS = 10


def load_fashion_mnist() -> (
    tuple[
        torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, DataLoader, DataLoader
    ]
):
    """Load Fashion-MNIST and return flat/image tensors + loaders.

    Returns:
        (X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader)
    """
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=True,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR),
        train=False,
        download=True,
        transform=torchvision.transforms.ToTensor(),
    )

    X_img = torch.stack([train_set[i][0] for i in range(len(train_set))])
    X_img = X_img.to(device).float()
    X_flat = X_img.reshape(len(X_img), -1)

    X_test_img = torch.stack([test_set[i][0] for i in range(len(test_set))])
    X_test_img = X_test_img.to(device).float()
    X_test_flat = X_test_img.reshape(len(X_test_img), -1)

    flat_loader = DataLoader(TensorDataset(X_flat), batch_size=256, shuffle=True)
    img_loader = DataLoader(TensorDataset(X_img), batch_size=256, shuffle=True)

    print(
        f"Fashion-MNIST loaded: {len(X_img)} train + {len(X_test_img)} test images, "
        f"shape {tuple(X_img.shape[1:])}, pixel range [{X_img.min():.2f}, {X_img.max():.2f}]"
    )

    return X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader


def get_fashion_mnist_labels() -> tuple[torch.Tensor, torch.Tensor]:
    """Return train and test label tensors."""
    train_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=True, download=True
    )
    test_set = torchvision.datasets.FashionMNIST(
        root=str(DATA_DIR), train=False, download=True
    )
    train_labels = torch.tensor([train_set[i][1] for i in range(len(train_set))])
    test_labels = torch.tensor([test_set[i][1] for i in range(len(test_set))])
    return train_labels, test_labels


# ════════════════════════════════════════════════════════════════════════
# KAILASH ENGINE SETUP
# ════════════════════════════════════════════════════════════════════════


async def _setup_engines():
    """Open kailash-ml 1.1.1 tracker + registry. 5-tuple preserved for callers."""
    # Schema-conflict workaround (kailash-ml 1.5.x): ExperimentTracker
    # and ModelRegistry use incompatible _kml_model_versions schemas.
    # Route them to separate sqlite files until upstream fixes the conflict.
    db = "sqlite:///mlfp05_autoencoders.db"
    registry_db = "sqlite:///mlfp05_autoencoders_registry.db"
    tracker = await ExperimentTracker.create(store_url=db)
    conn = ConnectionManager(registry_db)
    await conn.initialize()
    registry = ModelRegistry(conn)
    return conn, tracker, "m5_autoencoders", registry, True


def setup_engines() -> tuple:
    """Synchronously set up kailash-ml engines."""
    return asyncio.run(_setup_engines())


# ════════════════════════════════════════════════════════════════════════
# VISUALISATION UTILITIES — "Seeing Is Believing"
# ════════════════════════════════════════════════════════════════════════


def show_reconstruction(model, test_data, title, n=10, is_conv=False):
    """Show original vs reconstructed images side by side."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n].to(device)
        result = model(x)
        x_hat = result[0]

    fig, axes = plt.subplots(2, n, figsize=(15, 3))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n):
        if is_conv:
            orig = x[i].cpu().squeeze()
            recon = x_hat[i].cpu().squeeze()
        else:
            orig = x[i].cpu().reshape(28, 28)
            recon = x_hat[i].cpu().reshape(28, 28)

        axes[0, i].imshow(orig, cmap="gray")
        axes[0, i].axis("off")
        if i == 0:
            axes[0, i].set_title("Original", fontsize=9)

        axes[1, i].imshow(recon.clamp(0, 1), cmap="gray")
        axes[1, i].axis("off")
        if i == 0:
            axes[1, i].set_title("Reconstructed", fontsize=9)

    plt.tight_layout()
    fname = (
        OUTPUT_DIR
        / f"ex1_{title.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png"
    )
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_denoising_grid(model, clean_data, title, n=10, sigma=0.3):
    """3-row grid: original, noisy input, cleaned output."""
    model.eval()
    with torch.no_grad():
        clean = clean_data[:n].to(device)
        noisy = torch.clamp(clean + sigma * torch.randn_like(clean), 0.0, 1.0)
        result = model(noisy)
        cleaned = result[0]

    fig, axes = plt.subplots(3, n, figsize=(15, 4.5))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    row_labels = ["Original", "Noisy Input", "Cleaned Output"]

    for i in range(n):
        for row, data in enumerate([clean, noisy, cleaned]):
            img = data[i].cpu().reshape(28, 28)
            axes[row, i].imshow(img.clamp(0, 1), cmap="gray")
            axes[row, i].axis("off")
            if i == 0:
                axes[row, i].set_title(row_labels[row], fontsize=9)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_denoising_ae.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_activation_sparsity(model, test_data, title="Sparse AE Activations"):
    """Histogram of hidden-layer activations showing sparsity."""
    model.eval()
    with torch.no_grad():
        x = test_data[:1000].to(device)
        h = model.encoder(x)

    activations = h.cpu().numpy().flatten()

    _, ax = plt.subplots(1, 1, figsize=(8, 4))
    ax.hist(activations, bins=100, color="steelblue", edgecolor="white", alpha=0.8)
    ax.set_title(title, fontsize=14, fontweight="bold")
    ax.set_xlabel("Activation Value")
    ax.set_ylabel("Frequency")
    pct_near_zero = (np.abs(activations) < 0.1).mean() * 100
    ax.annotate(
        f"{pct_near_zero:.1f}% of activations near zero",
        xy=(0.95, 0.95),
        xycoords="axes fraction",
        ha="right",
        va="top",
        fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow"),
    )
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_sparse_activations.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_interpolation(model, test_data, title, n_steps=10, is_conv=False):
    """Morph between two images via latent space interpolation."""
    model.eval()
    with torch.no_grad():
        x1 = test_data[0:1].to(device)
        x2 = test_data[5:6].to(device)
        z1 = model.encoder(x1)
        z2 = model.encoder(x2)

        alphas = torch.linspace(0, 1, n_steps).to(device)
        interpolated = []
        for alpha in alphas:
            z = (1 - alpha) * z1 + alpha * z2
            x_hat = model.decoder(z)
            interpolated.append(x_hat)

    fig, axes = plt.subplots(1, n_steps + 2, figsize=(16, 2))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    src_img = x1[0].cpu().reshape(28, 28) if not is_conv else x1[0].cpu().squeeze()
    axes[0].imshow(src_img, cmap="gray")
    axes[0].set_title("Start", fontsize=8)
    axes[0].axis("off")

    for i, x_hat in enumerate(interpolated):
        img = x_hat[0].cpu()
        img = img.reshape(28, 28) if not is_conv else img.squeeze()
        axes[i + 1].imshow(img.clamp(0, 1), cmap="gray")
        axes[i + 1].set_title(f"{alphas[i]:.1f}", fontsize=7)
        axes[i + 1].axis("off")

    tgt_img = x2[0].cpu().reshape(28, 28) if not is_conv else x2[0].cpu().squeeze()
    axes[-1].imshow(tgt_img, cmap="gray")
    axes[-1].set_title("End", fontsize=8)
    axes[-1].axis("off")

    plt.tight_layout()
    fname = OUTPUT_DIR / f"ex1_{title.lower().replace(' ', '_')}.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_generated_samples(model, title="VAE Generated Samples", grid_size=8):
    """Grid of images sampled from the VAE's learned prior N(0, I)."""
    model.eval()
    n = grid_size * grid_size
    with torch.no_grad():
        samples = model.sample(n).cpu()

    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    fig.suptitle(title, fontsize=14, fontweight="bold")
    for i in range(grid_size):
        for j in range(grid_size):
            idx = i * grid_size + j
            axes[i, j].imshow(samples[idx].reshape(28, 28).clamp(0, 1), cmap="gray")
            axes[i, j].axis("off")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_generated_samples.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_latent_traversal(
    model, test_data, title="VAE Latent Traversal", n_dims=5, n_steps=11
):
    """Vary one latent dimension at a time and observe what changes."""
    model.eval()
    with torch.no_grad():
        x = test_data[0:1].to(device)
        mu, _ = model.encode(x)
        base_z = mu.clone()

    traversal_range = torch.linspace(-3, 3, n_steps)
    fig, axes = plt.subplots(n_dims, n_steps, figsize=(14, n_dims * 1.4))
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for dim in range(n_dims):
        for step_idx, val in enumerate(traversal_range):
            z = base_z.clone()
            z[0, dim] = val
            with torch.no_grad():
                x_hat = model.decoder(z)
            img = x_hat[0].cpu().reshape(28, 28).clamp(0, 1)
            axes[dim, step_idx].imshow(img, cmap="gray")
            axes[dim, step_idx].axis("off")
            if dim == 0:
                axes[dim, step_idx].set_title(f"z={val:.1f}", fontsize=7)
        axes[dim, 0].set_ylabel(f"dim {dim}", fontsize=8, rotation=0, labelpad=30)

    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_vae_latent_traversal.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


def show_timeseries_reconstruction(model, test_data, title, n_series=4):
    """Overlay original vs reconstructed time series."""
    model.eval()
    with torch.no_grad():
        x = test_data[:n_series].to(device)
        x_hat, _ = model(x)

    fig, axes = plt.subplots(n_series, 1, figsize=(14, 3 * n_series))
    if n_series == 1:
        axes = [axes]
    fig.suptitle(title, fontsize=14, fontweight="bold")

    for i in range(n_series):
        orig = x[i].cpu().numpy()
        recon = x_hat[i].cpu().numpy()
        t = np.arange(len(orig))

        axes[i].plot(t, orig, "b-", linewidth=1.5, label="Original", alpha=0.8)
        axes[i].plot(t, recon, "r--", linewidth=1.5, label="Reconstructed", alpha=0.8)
        axes[i].set_ylabel(f"Series {i + 1}")
        axes[i].legend(loc="upper right", fontsize=8)
        axes[i].grid(True, alpha=0.3)

    axes[-1].set_xlabel("Time Step")
    plt.tight_layout()
    fname = OUTPUT_DIR / "ex1_recurrent_ae_timeseries.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"  Saved: {fname}")


# ════════════════════════════════════════════════════════════════════════
# TRAINING LOOP — shared by all variants
# ════════════════════════════════════════════════════════════════════════

# Collect results across variants (populated by train_variant)
all_losses: dict[str, list[float]] = {}
all_models: dict[str, nn.Module] = {}


async def _train_variant_async(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Universal training loop for any AE variant."""
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses: list[float] = []

    params = {
        "model_type": name,
        "latent_dim": str(LATENT_DIM),
        "epochs": str(epochs),
        "lr": str(lr),
        "dataset_size": str(len(loader.dataset)),
        "batch_size": str(loader.batch_size),
    }
    if extra_params:
        params.update(extra_params)

    async with tracker.track(experiment=exp_name, run_name=name) as run:
        await run.log_params(params)

        for epoch in range(epochs):
            batch_losses = []
            for (xb,) in loader:
                opt.zero_grad()
                loss, _ = loss_fn(model, xb)
                loss.backward()
                opt.step()
                batch_losses.append(loss.item())
            epoch_loss = float(np.mean(batch_losses))
            losses.append(epoch_loss)
            await run.log_metric("loss", epoch_loss, step=epoch + 1)
            if (epoch + 1) % 5 == 0 or epoch == 0:
                print(f"  [{name}] epoch {epoch + 1}/{epochs}  loss={epoch_loss:.4f}")
        await run.log_metric("final_loss", losses[-1])

    return losses


def train_variant(
    tracker: ExperimentTracker,
    exp_name: str,
    model: nn.Module,
    name: str,
    loader: DataLoader,
    loss_fn,
    epochs: int = EPOCHS,
    lr: float = 1e-3,
    extra_params: dict | None = None,
) -> list[float]:
    """Sync wrapper for training with ExperimentTracker integration."""
    losses = asyncio.run(
        _train_variant_async(
            tracker, exp_name, model, name, loader, loss_fn, epochs, lr, extra_params
        )
    )
    all_losses[name] = losses
    all_models[name] = model
    return losses


# ════════════════════════════════════════════════════════════════════════
# MODEL REGISTRATION
# ════════════════════════════════════════════════════════════════════════


async def _register_model(
    registry: ModelRegistry,
    name: str,
    model: nn.Module,
    final_loss: float,
):
    """Register a single model variant."""
    from kailash_ml.types import MetricSpec

    model_bytes = pickle.dumps(model.state_dict())
    version = await registry.register_model(
        name=f"m5_{name}",
        artifact=model_bytes,
        metrics=[
            MetricSpec(name="final_loss", value=final_loss),
            MetricSpec(name="latent_dim", value=float(LATENT_DIM)),
            MetricSpec(name="epochs", value=float(EPOCHS)),
        ],
    )
    print(f"  Registered {name}: version={version.version}, loss={final_loss:.4f}")
    return version


def register_model(
    registry: ModelRegistry, name: str, model: nn.Module, final_loss: float
):
    """Sync wrapper for model registration."""
    return asyncio.run(_register_model(registry, name, model, final_loss))




# ════════════════════════════════════════════════════════════════════════
# MLFP05 — Exercise 1.2: Undercomplete Autoencoder (Bottleneck Compression)
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Build an undercomplete AE with bottleneck (784 -> 16 = 49:1 compression)
#   - Understand WHY forced compression solves the identity risk
#   - Visualise blurry but meaningful reconstructions
#   - Apply to credit card fraud detection at a Singapore bank
#   - Quantify business impact in S$ with precision-recall analysis
#
# PREREQUISITES: 01_standard_ae.py (identity risk understanding)
# ESTIMATED TIME: ~20 min
#
# TASKS:
#   1. Build undercomplete AE (784 -> 256 -> 64 -> 16)
#   2. Train on Fashion-MNIST and visualise reconstructions
#   3. Apply: fraud detection at a Singapore bank using anomaly reconstruction error
#   4. Business impact analysis with S$ projections
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset



## THEORY — Forced Compression via Bottleneck


In [ ]:
# The fix for the identity risk is simple: make the bottleneck SMALLER
# than the input. With latent_dim=16, the encoder must compress 784
# pixels into just 16 numbers — a 49:1 compression ratio.
#
# Analogy: A 50-page quarterly report compressed into a one-page
# executive summary. The summary MUST capture the key points because
# there is no room for everything. That forced compression is exactly
# what the undercomplete bottleneck does.
#
# The encoder must learn WHAT MATTERS in each image — the difference
# between a shirt and a shoe, not the exact pixel values. This is
# representation learning: extracting structure from data.



## TASK 1 — Load data and engines


In [ ]:
X_flat, X_test_flat, X_img, X_test_img, flat_loader, img_loader = load_fashion_mnist()
conn, tracker, exp_name, registry, has_registry = setup_engines()



## TASK 2 — Build and Train Undercomplete AE


Bottleneck forces compression: 784 -> 256 -> 64 -> 16 -> 64 -> 256 -> 784.


In [ ]:
class UndercompleteAE(nn.Module):

    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def undercomplete_ae_loss(model, xb):
    x_hat, _ = model(xb)
    return F.mse_loss(x_hat, xb), {}


print("\n" + "=" * 70)
print("  Undercomplete AE — Forced Compression (latent=16)")
print("=" * 70)
print("  784 pixels -> 16 numbers. Compression ratio 49:1.")

undercomplete_model = UndercompleteAE(INPUT_DIM, LATENT_DIM)
undercomplete_losses = train_variant(
    tracker,
    exp_name,
    undercomplete_model,
    "undercomplete_ae",
    flat_loader,
    undercomplete_ae_loss,
)



## DIAGNOSTIC CHECKPOINT — five instruments before Visualise


In [ ]:
# Same pattern as 01_standard_ae.py — see that file for the full
# Prescription Pad walkthrough. Here we expect a DIFFERENT picture:
# the undercomplete bottleneck (latent=16) blocks identity-copy, so
# gradients should be healthier across the encoder *while* the
# train loss stays noticeably higher than the overcomplete AE —
# that higher loss is the SIGNAL of genuine compression learning.
from kailash_ml.diagnostics import run_diagnostic_checkpoint


def _diag_loss(m, batch):
    xb = batch[0] if isinstance(batch, (tuple, list)) else batch
    loss, _ = undercomplete_ae_loss(m, xb)
    return loss


print("\n── Diagnostic Report (Undercomplete AE) ──")
diag, findings = run_diagnostic_checkpoint(
    undercomplete_model,
    flat_loader,
    _diag_loss,
    title="Undercomplete AE (latent=16)",
    n_batches=8,
    train_losses=undercomplete_losses,
    show=False,
)
print_prescription_pad(findings, "Undercomplete AE (latent=16)")

# ══════ READING THE PRESCRIPTION PAD (key: see 01_standard_ae.py) ══════
# Compare with 01: the 16-unit bottleneck cannot copy, so the final
# train loss should sit HIGHER than the overcomplete AE's — that gap is
# compression, not failure. On the pad, check the narrow encoder layers:
# vanishing gradients or a high dead-ReLU share there shrink the
# effective latent size even further.



## VISUALISE — Reconstruction grid


In [ ]:
show_reconstruction(
    undercomplete_model, X_test_flat, f"Undercomplete AE (latent={LATENT_DIM})"
)



### Checkpoint


In [ ]:
assert len(undercomplete_losses) == EPOCHS
assert undercomplete_losses[-1] < undercomplete_losses[0]
# INTERPRETATION: The reconstructions are blurry but recognisable.
# The model kept the SHAPE (is it a shirt? a shoe?) but lost DETAIL
# (exact button placement, stitching pattern). This is the information
# bottleneck principle: compress enough, and the model learns structure.
print("\n--- Checkpoint passed --- undercomplete AE trained\n")

if has_registry:
    register_model(
        registry, "undercomplete_ae", undercomplete_model, undercomplete_losses[-1]
    )



## APPLY — Credit Card Fraud Detection at a Singapore Bank


In [ ]:
# BUSINESS SCENARIO: You are a fraud analyst at a Singapore retail bank. 99.8% of
# daily transactions are legitimate. You have NO labelled fraud
# examples — only a gut feeling that "unusual" transactions deserve
# investigation. Your manager asks: "Can we catch more fraud without
# drowning investigators in false alerts?"
#
# TECHNIQUE: Train on ONLY normal transactions so the AE learns what
# "normal" looks like. At inference, legitimate transactions reconstruct
# well (low error); fraudulent ones reconstruct poorly (high error)
# because the encoder never learned their patterns.

print("\n" + "=" * 70)
print("  APPLICATION: Credit Card Fraud Detection (Singapore bank)")
print("=" * 70)

# --- Generate realistic Singapore bank transaction data ---
N_TOTAL = 200_000
FRAUD_RATE = 0.002  # 0.2% fraud — realistic for Singapore card-present

n_fraud = int(N_TOTAL * FRAUD_RATE)
n_normal = N_TOTAL - n_fraud
rng = np.random.default_rng(42)

# Normal transaction features
normal_amounts = rng.lognormal(mean=3.5, sigma=1.2, size=n_normal).clip(0.5, 5000)
normal_hour = rng.normal(loc=14, scale=4, size=n_normal).clip(0, 23).astype(int)
normal_merchant_cat = rng.choice(
    range(15),
    size=n_normal,
    p=[
        0.18,
        0.15,
        0.12,
        0.10,
        0.08,
        0.07,
        0.06,
        0.05,
        0.04,
        0.04,
        0.03,
        0.03,
        0.02,
        0.02,
        0.01,
    ],
)
normal_is_online = rng.binomial(1, 0.35, size=n_normal)
normal_distance = rng.exponential(scale=5, size=n_normal).clip(0, 50)
normal_freq_24h = rng.poisson(lam=2, size=n_normal)
normal_amt_ratio = rng.normal(1.0, 0.3, size=n_normal).clip(0.1, 3.0)
normal_foreign = rng.binomial(1, 0.08, size=n_normal)

# Fraud transaction features — shifted distributions
fraud_amounts = rng.lognormal(mean=5.5, sigma=1.5, size=n_fraud).clip(10, 50000)
fraud_hour = rng.choice([0, 1, 2, 3, 4, 22, 23], size=n_fraud)
fraud_merchant_cat = rng.choice(
    range(15),
    size=n_fraud,
    p=[
        0.02,
        0.02,
        0.03,
        0.03,
        0.05,
        0.05,
        0.05,
        0.08,
        0.10,
        0.10,
        0.12,
        0.12,
        0.08,
        0.08,
        0.07,
    ],
)
fraud_is_online = rng.binomial(1, 0.75, size=n_fraud)
fraud_distance = rng.exponential(scale=40, size=n_fraud).clip(0, 200)
fraud_freq_24h = rng.poisson(lam=8, size=n_fraud)
fraud_amt_ratio = rng.normal(4.0, 1.5, size=n_fraud).clip(0.5, 15.0)
fraud_foreign = rng.binomial(1, 0.45, size=n_fraud)

# Combine into polars DataFrame
amounts = np.concatenate([normal_amounts, fraud_amounts])
hours = np.concatenate([normal_hour, fraud_hour])
merchant_cats = np.concatenate([normal_merchant_cat, fraud_merchant_cat])
is_online = np.concatenate([normal_is_online, fraud_is_online])
distances = np.concatenate([normal_distance, fraud_distance])
freq_24h = np.concatenate([normal_freq_24h, fraud_freq_24h])
amt_ratios = np.concatenate([normal_amt_ratio, fraud_amt_ratio])
foreign = np.concatenate([normal_foreign, fraud_foreign])
labels = np.concatenate([np.zeros(n_normal), np.ones(n_fraud)])

df = pl.DataFrame(
    {
        "amount": amounts,
        "hour": hours,
        "merchant_category": merchant_cats,
        "is_online": is_online,
        "distance_from_home_km": distances,
        "transactions_last_24h": freq_24h,
        "amount_vs_avg_ratio": amt_ratios,
        "is_foreign": foreign,
        "is_fraud": labels,
    }
).sample(fraction=1.0, seed=42, shuffle=True)

print(
    f"Dataset: {df.shape[0]:,} transactions, {df.filter(pl.col('is_fraud') == 1).shape[0]} fraud ({FRAUD_RATE*100:.1f}%)"
)

# --- Prepare training data (normal-only) ---
feature_cols = [c for c in df.columns if c != "is_fraud"]
all_features = df.select(feature_cols).to_numpy().astype(np.float32)
all_labels = df["is_fraud"].to_numpy()

feat_min = all_features.min(axis=0)
feat_max = all_features.max(axis=0)
feat_range = feat_max - feat_min
feat_range[feat_range == 0] = 1.0
all_features_norm = (all_features - feat_min) / feat_range

normal_mask = all_labels == 0
fraud_mask = all_labels == 1
normal_features = all_features_norm[normal_mask]
fraud_features = all_features_norm[fraud_mask]

n_train = int(len(normal_features) * 0.8)
train_features = normal_features[:n_train]
test_normal = normal_features[n_train:]
test_fraud = fraud_features
test_features = np.vstack([test_normal, test_fraud])
test_labels = np.concatenate([np.zeros(len(test_normal)), np.ones(len(test_fraud))])

train_tensor = torch.tensor(train_features, device=device)
test_tensor = torch.tensor(test_features, device=device)
fraud_train_loader = DataLoader(
    TensorDataset(train_tensor), batch_size=512, shuffle=True
)

print(f"Training on {len(train_features):,} normal-only transactions")
print(f"Test set: {len(test_normal):,} normal + {len(test_fraud):,} fraud")

# --- Build and train fraud detector ---
FRAUD_INPUT_DIM = len(feature_cols)


class FraudDetectorAE(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


fraud_model = FraudDetectorAE(FRAUD_INPUT_DIM, 3).to(device)
fraud_opt = torch.optim.Adam(fraud_model.parameters(), lr=1e-3)

print("\nTraining fraud detection autoencoder...")
for epoch in range(50):
    fraud_model.train()
    epoch_loss = 0.0
    n_batches = 0
    for (batch,) in fraud_train_loader:
        recon = fraud_model(batch)
        loss = F.mse_loss(recon, batch)
        fraud_opt.zero_grad()
        loss.backward()
        fraud_opt.step()
        epoch_loss += loss.item()
        n_batches += 1
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}/50: loss = {epoch_loss/n_batches:.6f}")

# --- Compute reconstruction errors ---
fraud_model.eval()
with torch.no_grad():
    recon_test = fraud_model(test_tensor)
    errors = ((test_tensor - recon_test) ** 2).mean(dim=1).cpu().numpy()

normal_errors = errors[test_labels == 0]
fraud_errors = errors[test_labels == 1]

print(f"\nReconstruction error statistics:")
print(
    f"  Normal: mean={normal_errors.mean():.6f}, p95={np.percentile(normal_errors, 95):.6f}"
)
print(
    f"  Fraud:  mean={fraud_errors.mean():.6f}, p95={np.percentile(fraud_errors, 95):.6f}"
)
print(f"  Separation ratio: {fraud_errors.mean() / normal_errors.mean():.1f}x")

# --- Visualisation 1: Error distributions ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(
    normal_errors, bins=80, alpha=0.7, label="Normal", color="#2196F3", density=True
)
axes[0].hist(
    fraud_errors, bins=80, alpha=0.7, label="Fraud", color="#F44336", density=True
)
axes[0].axvline(
    np.percentile(normal_errors, 95),
    color="#FF9800",
    linestyle="--",
    label=f"95th pctl = {np.percentile(normal_errors, 95):.4f}",
)
axes[0].set_xlabel("Reconstruction Error (MSE)")
axes[0].set_ylabel("Density")
axes[0].set_title("Reconstruction Error Distribution\nNormal vs Fraud", fontsize=13)
axes[0].legend(fontsize=10)

bp = axes[1].boxplot(
    [normal_errors, fraud_errors],
    tick_labels=["Normal", "Fraud"],
    patch_artist=True,
    widths=0.5,
)
bp["boxes"][0].set_facecolor("#2196F3")
bp["boxes"][1].set_facecolor("#F44336")
axes[1].set_ylabel("Reconstruction Error (MSE)")
axes[1].set_title("Error Comparison: Normal vs Fraud", fontsize=13)
plt.tight_layout()
plt.savefig(
    OUTPUT_DIR / "ex1_fraud_error_distribution.png", dpi=150, bbox_inches="tight"
)
plt.show()

# --- Visualisation 2: Precision-Recall ---
thresholds = np.linspace(errors.min(), np.percentile(errors, 99.5), 200)
precisions, recalls, f1_scores = [], [], []
for t in thresholds:
    predicted_fraud = errors > t
    true_fraud = test_labels == 1
    tp = np.sum(predicted_fraud & true_fraud)
    fp = np.sum(predicted_fraud & ~true_fraud)
    fn = np.sum(~predicted_fraud & true_fraud)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    precisions.append(precision)
    recalls.append(recall)
    f1_scores.append(f1)

precisions = np.array(precisions)
recalls = np.array(recalls)
f1_scores = np.array(f1_scores)
best_f1_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_f1_idx]
best_precision = precisions[best_f1_idx]
best_recall = recalls[best_f1_idx]
best_f1 = f1_scores[best_f1_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(recalls, precisions, color="#673AB7", linewidth=2)
axes[0].scatter(
    [best_recall],
    [best_precision],
    color="#F44336",
    s=100,
    zorder=5,
    label=f"Best F1={best_f1:.3f}\n(P={best_precision:.3f}, R={best_recall:.3f})",
)
axes[0].set_xlabel("Recall (Fraud Caught)")
axes[0].set_ylabel("Precision (True Among Flagged)")
axes[0].set_title("Precision-Recall Curve", fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, f1_scores, color="#009688", linewidth=2, label="F1 Score")
axes[1].plot(
    thresholds, precisions, color="#2196F3", linewidth=1.5, alpha=0.7, label="Precision"
)
axes[1].plot(
    thresholds, recalls, color="#F44336", linewidth=1.5, alpha=0.7, label="Recall"
)
axes[1].axvline(
    best_threshold,
    color="#FF9800",
    linestyle="--",
    label=f"Optimal threshold = {best_threshold:.5f}",
)
axes[1].set_xlabel("Reconstruction Error Threshold")
axes[1].set_ylabel("Score")
axes[1].set_title("Threshold Selection", fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_fraud_precision_recall.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Visualisation 3: Top anomalies ---
top_k = 20
top_indices = np.argsort(errors)[-top_k:][::-1]
fig, ax = plt.subplots(figsize=(12, 6))
colors = ["#F44336" if test_labels[i] == 1 else "#2196F3" for i in top_indices]
ax.barh(range(top_k), errors[top_indices], color=colors)
ax.set_yticks(range(top_k))
ax.set_yticklabels(
    [f"Txn #{i} ({'FRAUD' if test_labels[i]==1 else 'Normal'})" for i in top_indices],
    fontsize=9,
)
ax.set_xlabel("Reconstruction Error (Anomaly Score)")
ax.set_title(
    f"Top {top_k} Most Anomalous Transactions\nRed = True Fraud, Blue = Normal",
    fontsize=13,
)
ax.axvline(
    best_threshold,
    color="#FF9800",
    linestyle="--",
    linewidth=2,
    label=f"Detection threshold = {best_threshold:.5f}",
)
ax.legend(fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "ex1_fraud_top_anomalies.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Business Impact Analysis ---
BANK_DAILY_TRANSACTIONS = 2_000_000  # illustrative scenario figures
AVG_FRAUD_VALUE_SGD = 800
RULE_BASED_RECALL = 0.67
DAILY_FRAUD_COUNT = int(BANK_DAILY_TRANSACTIONS * FRAUD_RATE)
FPR_AT_BEST = np.sum((errors > best_threshold) & (test_labels == 0)) / np.sum(
    test_labels == 0
)

daily_fraud_caught_ae = int(DAILY_FRAUD_COUNT * best_recall)
daily_fraud_caught_rules = int(DAILY_FRAUD_COUNT * RULE_BASED_RECALL)
daily_additional_caught = daily_fraud_caught_ae - daily_fraud_caught_rules
daily_false_alerts = int(BANK_DAILY_TRANSACTIONS * (1 - FRAUD_RATE) * FPR_AT_BEST)
daily_value_saved = daily_additional_caught * AVG_FRAUD_VALUE_SGD
annual_value_saved = daily_value_saved * 365

print("\n" + "=" * 64)
print("BUSINESS IMPACT SUMMARY — Card Fraud Detection (illustrative bank)")
print("=" * 64)
print(f"\nDaily card transactions:         {BANK_DAILY_TRANSACTIONS:>12,}")
print(f"Estimated daily fraud events:    {DAILY_FRAUD_COUNT:>12,}")
print(f"Average fraud value:             {'S$' + str(AVG_FRAUD_VALUE_SGD):>12}")
print(f"\nCurrent rule-based system:")
print(f"  Fraud recall:                  {RULE_BASED_RECALL:>11.0%}")
print(f"  Fraud caught/day:              {daily_fraud_caught_rules:>12,}")
print(f"\nAutoencoder-based system (optimal threshold = {best_threshold:.5f}):")
print(f"  Fraud recall:                  {best_recall:>11.1%}")
print(f"  Precision:                     {best_precision:>11.1%}")
print(f"  Fraud caught/day:              {daily_fraud_caught_ae:>12,}")
print(f"  False alerts/day:              {daily_false_alerts:>12,}")
print(f"\nIncremental impact:")
print(f"  Additional fraud caught/day:   {daily_additional_caught:>12,}")
print(f"  Value saved per day:           {'S$' + f'{daily_value_saved:,.0f}':>12}")
print(f"  Value saved per year:          {'S$' + f'{annual_value_saved:,.0f}':>12}")
print("=" * 64)



## REFLECTION


[x] Built an undercomplete AE with 49:1 compression (784 -> 16)
  [x] Observed blurry but meaningful reconstructions — structure preserved
  [x] Applied bottleneck AE to credit card fraud detection at a Singapore bank
  [x] Computed precision-recall curves for threshold selection
  [x] Quantified business impact: S$ value of additional fraud prevented

  KEY INSIGHT: The bottleneck forces the encoder to learn what MATTERS.
  A shirt's overall shape is preserved; its button stitching is lost.
  In fraud detection, normal transaction PATTERNS are preserved;
  fraudulent patterns (unusual amount + time + merchant) cannot be
  reconstructed, producing the anomaly signal.

  Next: 03_denoising_ae.py adds noise robustness...


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

